# Multi-camera person Re-ID and face association: Colab demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbderrahimZineddineEsi/multi-modal-person-reid/blob/v2/notebooks/colab_demo.ipynb)

This notebook installs the `reid` package and drives it through its command line. It contains no pipeline logic of its own: every step is a `reid ...` command or a call to a documented package function, so what you see here is what you get from a terminal.

What it does, in order:

1. checks the GPU and installs the package;
2. mounts Google Drive (all results live under one `DATA_DIR`, so they survive the runtime);
3. gets footage: a labelled AI City 2024 scene, your own videos, or virtual cameras cut from one video;
4. processes the cameras into one shared identity space (`--preset balanced`);
5. evaluates against the ground truth (AI City only);
6. renders a mosaic of all cameras and a clip that follows one person across cameras, and searches by image;
7. launches the Gradio app.

Use a **T4 GPU runtime** (Runtime > Change runtime type). Face and body weights are downloaded at runtime and are licensed for non-commercial research use (see the README).

## 1. GPU and installation

First check that a GPU is attached. The `balanced` preset needs about 1 GB of VRAM and runs on a free T4.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

Set the branch or tag to install with `REID_REF`.

The install keeps Colab's preinstalled PyTorch (it is not re-downloaded) and takes about 2-4 minutes. Its log is written to `/content/pip_install.log`; the cell prints the last lines if something fails.

In [ ]:
import os

REID_REF = "v2"  # branch or tag of the repository to install
os.environ["REID_REF"] = REID_REF

In [ ]:
%%bash
set -o pipefail
python -c "import torch; print('preinstalled torch', torch.__version__, '| CUDA', torch.version.cuda)"
pip install --progress-bar off "reid[models,app,eval] @ git+https://github.com/AbderrahimZineddineEsi/multi-modal-person-reid@${REID_REF}" \
  > /content/pip_install.log 2>&1 || { tail -n 40 /content/pip_install.log; exit 1; }
tail -n 3 /content/pip_install.log
python -c "import torch; print('torch after install', torch.__version__)"

### Fixing the OpenCV / ONNX Runtime clash

`ultralytics` pulls in `opencv-python` and `insightface` pulls in `onnxruntime`. They ship the same Python modules as `opencv-python-headless` and `onnxruntime-gpu`, which `reid` needs, so pip can leave a broken mix of files behind. The cell below removes the clashing wheels and force-reinstalls the two we want (without touching their dependencies). It is harmless if nothing clashed.

If a later cell fails with an `ImportError` mentioning `numpy` or `cv2`, use **Runtime > Restart session** and run the notebook again from the top (Drive stays mounted).

In [ ]:
%%bash
pip uninstall -y -q opencv-python onnxruntime
pip install -q --force-reinstall --no-deps opencv-python-headless "onnxruntime-gpu==1.32.0"
python -c "import cv2, onnxruntime; print('opencv', cv2.__version__, '| onnxruntime', onnxruntime.__version__)"

In [ ]:
!reid --help | head -20

## 2. Where to keep the data

Everything (database, vectors, crops, renders, evaluation results, downloaded datasets) goes under `DATA_DIR`. On Colab that should be a Drive folder so a disconnected runtime does not lose a processing run. If you decline the Drive prompt, the notebook falls back to local storage with a warning.

In [ ]:
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/reid_data")
try:
    from google.colab import drive

    drive.mount("/content/drive")
except Exception as exc:  # not on Colab, or the prompt was declined
    DATA_DIR = Path("/content/reid_data")
    print(f"WARNING: Drive not mounted ({exc!r}).")
    print(f"Using {DATA_DIR}: everything is lost when this runtime ends.")
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_DIR =", DATA_DIR)

## 3. Choose the footage

Run **one** of the three cells (a), (b) or (c). Each one sets `SOURCE` and `CAMERAS_YAML`, the camera set that the next steps use. A camera set lists the videos; all cameras of a set are processed into one shared identity space, and `start_ts` aligns them in time (0 on every camera means synchronized).

### (a) AI City 2024 scene with ground truth

`reid dataset fetch` downloads one retail scene (075) from the NVIDIA `PhysicalAI-SmartSpaces` dataset on Hugging Face (CC-BY-4.0, synthetic people, no login needed). It picks 4 cameras whose views overlap, keeps the first 2 minutes, downscales to 720p, converts the ground truth to a uniform format and writes the camera YAML with all cameras synchronized and no face cameras. Only the ground truth file (about 50 MB) and the selected cameras are fetched; `--stream` makes ffmpeg read just the first minutes instead of the whole videos.

In [ ]:
SOURCE = "aicity"
AICITY_DIR = DATA_DIR / "datasets" / "aicity24_075"
CAMERAS_YAML = DATA_DIR / "cameras_aicity24_075.yaml"
GT_CSV = AICITY_DIR / "gt.csv"

!reid dataset fetch aicity24 --scene 075 --cameras 4 --minutes 2 --height 720 --stream \
    --out {AICITY_DIR} --yaml {CAMERAS_YAML}

### (b) Your own videos from Drive

List your videos and write the camera YAML with `write_camera_yaml`. Put the videos of one place and one time span together: the more the views overlap in time and space, the more the identity association has to work with. Cameras with a clear view of faces can be flagged in `face_cams`; set `start_ts` per camera (epoch seconds of frame 0) if the recordings are not synchronized.

In [ ]:
from reid.datasets import write_camera_yaml

SOURCE = "own"
GT_CSV = None  # no ground truth: the evaluation step is skipped
VIDEOS = [  # edit: your own videos on Drive
    DATA_DIR / "videos" / "entrance.mp4",
    DATA_DIR / "videos" / "aisle.mp4",
]
CAMERAS_YAML = write_camera_yaml(
    DATA_DIR / "cameras_own.yaml",
    VIDEOS,
    face_cams=["cam_01"],  # ids are cam_01, cam_02, ... in list order
)
print(CAMERAS_YAML.read_text())

### (c) Virtual cameras from a single video

If you only have one video, `reid virtual-cams` derives several "cameras" from it (overlapping crops, optional mirror and time offset) and writes the camera YAML. This exercises the whole pipeline, but it does **not** show that the system generalizes across real viewpoints: the views share lighting and the overlap is the same pixels.

In [ ]:
SOURCE = "virtual"
GT_CSV = None
VIDEO = DATA_DIR / "videos" / "store.mp4"  # edit: one video on Drive
VIRTUAL_DIR = DATA_DIR / "virtual_cams"
CAMERAS_YAML = VIRTUAL_DIR / "cameras.yaml"

!reid virtual-cams {VIDEO} --out-dir {VIRTUAL_DIR} --n 3 --overlap 0.5 --mirror 2 \
    --width 960 --end 120 --yaml {CAMERAS_YAML}

## 4. Process the cameras

`reid process` detects and tracks people in every camera, builds tracklets (quality-gated body samples, faces on face cameras), and associates the tracklets into global identities. The `balanced` preset is the default for a T4: YOLO26m-pose detector, BoT-SORT tracker, CLIP-ReID body model, buffalo_l faces.

`STRIDE = 2` processes every second frame, which halves the time with little effect on tracking at 30 fps. Set it to 1 for the full frame rate. Each source gets its own `RUN_DIR`, so runs do not mix. The first run also downloads the model weights.

In [ ]:
STRIDE = 2
RUN_DIR = DATA_DIR / f"run_{SOURCE}"

!reid process --cameras {CAMERAS_YAML} --preset balanced --data-dir {RUN_DIR} \
    --set runtime.frame_stride={STRIDE}

The JSON above holds the run summary: number of tracklets and identities, how many tracklets were identified, tracklets per identity (1.0 means no fragmentation), and processing speed per stage. `reid info` shows what is stored.

In [ ]:
!reid info --preset balanced --data-dir {RUN_DIR} | tail -20

## 5. Evaluate against the ground truth (AI City only)

`reid evaluate` compares the stored boxes and gids with the ground truth and reports per-camera IDF1 / MOTA / ID switches, the cross-camera **CityFlow-style combined IDF1** (one accumulator over all cameras, so a person must keep one gid on every camera), fragmentation and purity, and held-out retrieval (Rank-1 / mAP from ground-truth crops that were not used to build the gallery). The result is also saved under `RUN_DIR/eval/`.

The data is synthetic, so use it to check that the pipeline logic and the cross-camera association work, not to tune absolute body-ReID thresholds.

In [ ]:
if SOURCE == "aicity":
    !reid evaluate --data-dir {RUN_DIR} --preset balanced --gt {GT_CSV}
else:
    print("No ground truth for this source: skipping the evaluation.")

## 6. Render videos and follow one person

All per-frame boxes are stored, so any video can be rendered after processing. `reid render --camera all --mosaic` writes each camera with its identity boxes and a synchronized mosaic of all cameras into `RUN_DIR/renders/`. `reid journey --gid G` prints where identity G was seen and renders a clip that follows that person from camera to camera.

Pick the identity that was seen in the most tracklets.

In [ ]:
!reid render --camera all --mosaic --scale 0.5 --data-dir {RUN_DIR} --preset balanced

In [ ]:
from reid.storage import db, files

conn = db.connect(files.db_path(RUN_DIR))
GID = max(db.list_identities(conn), key=lambda i: i.n_tracklets).gid
conn.close()
print("following identity", GID)

!reid journey --gid {GID} --data-dir {RUN_DIR} --preset balanced

In [ ]:
from IPython.display import Video, display


def show(path, width=800):
    path = Path(path)
    small = path.stat().st_size < 30e6  # embed small files; large ones would bloat the notebook
    if not small:
        print(
            f"{path} is {path.stat().st_size / 1e6:.0f} MB: too large to embed, open it on Drive."
        )
        return
    display(Video(str(path), embed=True, width=width))


show(RUN_DIR / "renders" / "mosaic.mp4")
show(RUN_DIR / "renders" / f"journey_gid{GID}.mp4")

## 7. Search by image

`reid search` embeds a query image and looks it up in the identity space (body, face, or both; a face match wins). Here the query is the best saved crop of the identity followed above, which is a circular test meant only to show the output; use any photo of a person instead. The status is `MATCH`, `AMBIGUOUS` (two identities too close) or `UNKNOWN`. Pass the same `--preset` as for processing so the query is embedded by the same model.

In [ ]:
conn = db.connect(files.db_path(RUN_DIR))
crop = next(t.best_crop_path for t in db.get_tracklets(conn, gid=GID) if t.best_crop_path)
conn.close()
QUERY = RUN_DIR / crop

!reid search --image {QUERY} --mode both --preset balanced --data-dir {RUN_DIR}

In [ ]:
from IPython.display import Image

Image(str(QUERY), width=120)

## 8. Launch the app

The Gradio app has tabs for processing, exploring identities and journeys, searching, per-camera views and metrics. `--share` opens a public `gradio.live` link, so it requires a login: the cell generates a password and prints it. Destructive admin actions stay hidden unless you add `--admin`.

In [ ]:
import secrets
import subprocess

PASSWORD = secrets.token_urlsafe(9)
print(f"Login: user 'reid', password '{PASSWORD}'")
cmd = [
    "reid",
    "app",
    "--share",
    "--auth",
    f"reid:{PASSWORD}",
    "--preset",
    "balanced",
    "--data-dir",
    str(RUN_DIR),
]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:  # stream the output: Gradio prints the public link while it runs
    print(line, end="")
proc.wait()